# 03 Inference

## 1. Data

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_selected_cleaned.csv"
TAB_DIR = ROOT / "outputs" / "tables"
SUM_DIR = ROOT / "outputs" / "summary"

BEHAVIOR_VAR = "EverCigaretteUse"
BEHAVIOR_P0 = 0.50
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0

processed = pd.read_csv(PROCESSED_PATH)
print("Processed data shape:", processed.shape)

Processed data shape: (14041, 3)


## 2.Proportion Inference: `EverCigaretteUse`

In [4]:
behavior_binary = processed["behavior_binary"].dropna().astype(int)
n_b = int(behavior_binary.shape[0])
x_b = int((behavior_binary == 1).sum())
phat = x_b / n_b

ci_b = stats.binomtest(x_b, n_b).proportion_ci(confidence_level=0.95, method="wilson")
z_stat = (phat - BEHAVIOR_P0) / np.sqrt(BEHAVIOR_P0 * (1 - BEHAVIOR_P0) / n_b)
p_value_b = 2 * (1 - stats.norm.cdf(abs(z_stat)))
p_value_display = "< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}"

prop_results = pd.DataFrame({
    "quantity": [
        "sample_size", "successes", "sample_proportion",
        "benchmark_p0", "z_statistic", "p_value", "ci_low", "ci_high"
    ],
    "value": [
        str(n_b),
        str(x_b),
        f"{phat:.4f}",
        f"{BEHAVIOR_P0:.4f}",
        f"{z_stat:.4f}",
        p_value_display,
        f"{ci_b.low:.4f}",
        f"{ci_b.high:.4f}"
    ]
})

decision_b = "reject H0" if p_value_b < 0.05 else "fail to reject H0"

display(prop_results)

#Save
behavior_summary = pd.DataFrame({
    "analysis": ["Population proportion"],
    "variable": [BEHAVIOR_VAR],
    "estimate": [f"{phat:.4f}"],
    "benchmark": [f"{BEHAVIOR_P0:.4f}"],
    "test_statistic": [f"{z_stat:.4f}"],
    "p_value": [
        "< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}"
    ],
    "ci_low": [f"{ci_b.low:.4f}"],
    "ci_high": [f"{ci_b.high:.4f}"],
    "decision_5pct": [decision_b]
})

behavior_summary.to_csv(TAB_DIR / "03_behavior_inference_summary.csv", index=False)

,quantity,value
0,sample_size,13601
1,successes,7164
2,sample_proportion,0.5267
3,benchmark_p0,0.5000
4,z_statistic,6.2337
5,p_value,< 0.0001
6,ci_low,0.5183
7,ci_high,0.5351


### Interpretation (Proportion Analysis)

- We used the sample to estimate and test whether the population proportion for **EverCigaretteUse** differs from the benchmark of **0.50**.
- The sample proportion is **0.5267**, so the observed proportion is slightly higher than **0.50** in this sample.
- Because the z-statistic (**6.2337**) is far larger than the two-sided critical value of **1.96**, we reject the null hypothesis at the 5% significance level.
- Since the benchmark proportion of 0.50 is not contained in the 95% confidence interval, the null hypothesis is rejected at the 5% significance level.
- Because the **p-value is < 0.0001**, we reject the null hypothesis at the 5% significance level and conclude that the population proportion is significantly above **0.50**.
- This is consistent with the EDA, where the success category appeared slightly more common than the failure category.
- A cautious point is that the result is statistically significant, but the difference is only about **0.0267** above the benchmark, so the effect is not large in practical size.